## Problem 1

In [1]:
import pulp as lp

### Model, Variables, Objective Functions and Constraints

In [6]:
# Data
fixed = [1000, 950, 875, 850, 800, 700]
variable = [21, 23, 25, 24, 20, 26]
capacity = [500, 600, 750, 400, 600, 800]
machines = range(6)

# Model
model = lp.LpProblem("Radford_Castings", lp.LpMinimize)

# Variables
x = lp.LpVariable.dicts("x", machines, lowBound=0)
y = lp.LpVariable.dicts("y", machines, lowBound=0, upBound=1, cat="Binary")

# Objective
model += lp.lpSum(fixed[i] * y[i] + variable[i] * x[i] for i in machines)

# Demand constraint
model += lp.lpSum(x[i] for i in machines) == 1800

# Capacity constraints
for i in machines:
    model += x[i] <= capacity[i] * y[i]

### Solve

In [7]:
# Solve
model.solve(lp.PULP_CBC_CMD(msg=False))

# Print results
print("Status:", lp.LpStatus[model.status])
for i in machines:
    print(f"Machine {i+1}: y={int(y[i].value())}, x={x[i].value()}")
print("Total Cost:", lp.value(model.objective))

Status: Optimal
Machine 1: y=1, x=500.0
Machine 2: y=1, x=600.0
Machine 3: y=0, x=0.0
Machine 4: y=1, x=100.0
Machine 5: y=1, x=600.0
Machine 6: y=0, x=0.0
Total Cost: 42300.0


## Problem 2 

### Data

In [11]:
sites = ["Sanford","Altamonte","Apopka","Casselberry","Maitland"]

cost = {
    "Sanford": 450,
    "Altamonte": 650,
    "Apopka": 550,
    "Casselberry": 500,
    "Maitland": 525
}

# Coverage from the PDF's X table
coverage = {
    1: ["Sanford","Apopka"],
    2: ["Sanford","Altamonte","Casselberry","Maitland"],
    3: ["Altamonte","Casselberry"],
    4: ["Apopka","Maitland"],
    5: ["Sanford","Altamonte"],
    6: ["Apopka","Maitland"],
    7: ["Casselberry","Maitland"]
}


### Model, Variables, Objective Functions and Constraints

In [12]:
model = lp.LpProblem("HCSF_Set_Covering", lp.LpMinimize)

# Decision variables: y_s = 1 if we build at site s
y = lp.LpVariable.dicts("Build", sites, lowBound=0, upBound=1, cat="Binary")

# Objective: minimize cost
model += lp.lpSum(cost[s] * y[s] for s in sites)

# Coverage constraints: every region must have ≥1 covering site selected
for r in coverage:
    model += lp.lpSum(y[s] for s in coverage[r]) >= 1, f"Cover_region_{r}"


### Solve

In [14]:
model.solve(lp.PULP_CBC_CMD(msg=False))

print("Status:", lp.LpStatus[model.status])
print("\nSelected sites:")
for s in sites:
    if y[s].value() == 1:
        print(s)

print("\nTotal Cost (in $1000s):", lp.value(model.objective))

Status: Optimal

Selected sites:
Sanford
Casselberry
Maitland

Total Cost (in $1000s): 1475.0


## Problem 3

### Data

In [15]:
sources = ["TX", "OK", "PA", "AL"]

avail = {
    "TX": 1500,
    "OK": 2000,
    "PA": 1500,
    "AL": 1800
}

# production per barrel
gas =   {"TX":2.0, "OK":1.8, "PA":2.3, "AL":2.1}
ker =   {"TX":2.8, "OK":2.3, "PA":2.2, "AL":2.6}
heat =  {"TX":1.7, "OK":1.75,"PA":1.6,"AL":1.9}
asp =   {"TX":2.4, "OK":1.9, "PA":2.6, "AL":2.4}

# purchase price per barrel
price = {"TX":22, "OK":21, "PA":22, "AL":23}

# trucking cost for sending truck to pick up from that state
truck = {"TX":1500, "OK":1700, "PA":1500, "AL":1400}

# minimum purchase requirement
min_order = 500

# truck capacity
truck_cap = 2000

# required production
req_gas = 750
req_ker = 800
req_heat = 1000
req_asp = 300

### Model, Variables, Objective Functions and Constraints

In [16]:
model = lp.LpProblem("Clampett_Oil", lp.LpMinimize)

# Decision variables
x = lp.LpVariable.dicts("Barrels", sources, lowBound=0)            # amount purchased
y = lp.LpVariable.dicts("UseTruck", sources, cat="Binary")          # 1 if truck goes to supplier

# Objective: purchase cost + trucking cost (truck cost only if y_s = 1)
model += lp.lpSum(price[s] * x[s] + truck[s] * y[s] for s in sources)

# Minimum order and availability + linking x ≤ avail*y
for s in sources:
    model += x[s] >= min_order * y[s]
    model += x[s] <= avail[s] * y[s]

# Total barrels cannot exceed truck capacity
model += lp.lpSum(x[s] for s in sources) <= truck_cap

# Production requirements
model += lp.lpSum(gas[s]  * x[s] for s in sources) >= req_gas
model += lp.lpSum(ker[s]  * x[s] for s in sources) >= req_ker
model += lp.lpSum(heat[s] * x[s] for s in sources) >= req_heat
model += lp.lpSum(asp[s]  * x[s] for s in sources) >= req_asp

### Solve

In [17]:
model.solve(lp.PULP_CBC_CMD(msg=False))

print("Status:", lp.LpStatus[model.status])

print("\nOptimal purchase plan:")
for s in sources:
    print(f"{s}: {x[s].value():.1f} barrels (truck used = {int(y[s].value())})")

print("\nTotal Cost = $", lp.value(model.objective))

Status: Optimal

Optimal purchase plan:
TX: 0.0 barrels (truck used = 0)
OK: 0.0 barrels (truck used = 0)
PA: 0.0 barrels (truck used = 0)
AL: 526.3 barrels (truck used = 1)

Total Cost = $ 13505.26317
